# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [1]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 5.30 MiB/s, done.
Resolving deltas: 100% (93/93), done.
--2026-07-05 13:29:52--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-05T14%3A07%3A24Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b

In [2]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [3]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:54,  1.15it/s]


In [4]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


--2026-07-05 13:37:02--  https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.87, 18.65.14.125, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-05 13:37:02 ERROR 404: Not Found.

tar: This does not look like a tar archive
tar: Exiting with failure status due to previous errors


0it [00:00, ?it/s]


 99%|█████████▉| 119/120 [00:39<00:00,  3.04it/s]


In [5]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-07-05 13:38:47--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 18.65.14.125, 18.65.14.87, 18.65.14.100, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.125|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&Expires=1783262327&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DY

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [6]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [7]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hi colmo"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [61]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.8 MB/s eta 0:00:00


In [56]:
!find ./my_custom_model/hi_colmo -name "*features*.npy" -delete

In [43]:
import yaml

with open("my_model.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

for k, v in config.items():
    if any(x in k.lower() for x in ["output", "feature", "clip", "path", "dir", "model"]):
        print(k, ":", v)

background_paths : ['./audioset_16k', './fma']
background_paths_duplication_rate : [1]
false_positive_validation_data_path : validation_set_features.npy
feature_data_files : {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
model_name : hi_colmo
model_type : dnn
output_dir : ./my_custom_model
piper_sample_generator_path : ./piper-sample-generator
rir_paths : ['./mit_rirs']


In [37]:
import glob
import soundfile as sf

bad = []
checked = 0

for path in glob.glob("**/*.wav", recursive=True):
    # 只检查 openWakeWord 训练相关目录，避免扫描太多外部数据
    if not any(k in path for k in [
        "training_tutorial_data",
        "generated",
        "positive",
        "negative",
        "adversarial",
    ]):
        continue

    try:
        info = sf.info(path)
        checked += 1
        if info.samplerate != 16000:
            bad.append((path, info.samplerate, info.duration))
    except Exception as e:
        print("read failed:", path, e)

print("checked:", checked)
print("bad:", len(bad))

for item in bad[:50]:
    print(item)

checked: 4022
bad: 4000
('my_custom_model/hi_colmo/positive_train/6e8bdcbc177f402a95b30cc2607e8e4e.wav', 22050, 0.626938775510204)
('my_custom_model/hi_colmo/positive_train/b7f038c5b22a4254bebf4f467b2b2c4e.wav', 22050, 0.6501587301587302)
('my_custom_model/hi_colmo/positive_train/d563ec4d1c75496a879a7a0f8aac42dd.wav', 22050, 0.603718820861678)
('my_custom_model/hi_colmo/positive_train/da78c71054d84e77a316404662b9a594.wav', 22050, 0.6385487528344671)
('my_custom_model/hi_colmo/positive_train/1d25c68d21144b528ac7a74da346fa02.wav', 22050, 0.6385487528344671)
('my_custom_model/hi_colmo/positive_train/5942802d01884f919cdd7add42c9ea14.wav', 22050, 0.6733786848072563)
('my_custom_model/hi_colmo/positive_train/e663cbc5b6664c62ac05ce8d11ac3604.wav', 22050, 0.6501587301587302)
('my_custom_model/hi_colmo/positive_train/75dc76a5514245eead22da0c9ca6f472.wav', 22050, 0.5688888888888889)
('my_custom_model/hi_colmo/positive_train/5b779375c82c49cfb01298cf2d1239ea.wav', 22050, 0.4876190476190476)
('my_c

In [38]:
!pip install -q librosa soundfile

In [39]:
import librosa
import soundfile as sf

fixed = 0

for path, old_sr, duration in bad:
    audio, sr = librosa.load(path, sr=16000, mono=True)
    sf.write(path, audio, 16000)
    fixed += 1

print("fixed:", fixed)

fixed: 4000


In [40]:
bad2 = []

for path in glob.glob("**/*.wav", recursive=True):
    if not any(k in path for k in [
        "training_tutorial_data",
        "generated",
        "positive",
        "negative",
        "adversarial",
    ]):
        continue

    try:
        info = sf.info(path)
        if info.samplerate != 16000:
            bad2.append((path, info.samplerate))
    except:
        pass

print("remaining bad:", len(bad2))
print(bad2[:20])

remaining bad: 0
[]


In [36]:
import soundfile as sf
import glob

files = [f for f in glob.glob("**/*.wav", recursive=True) if "positive" in f][:10]
for f in files:
    info = sf.info(f)
    print(f, info.samplerate)

openwakeword/notebooks/training_tutorial_data/positive/f58bffb52fe34301ac5f1bb7ef21d38a_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/54fb4b7410a2488898dbd36807101e9d_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/4c513dc7d1304df1b5bf8925b220e315_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/4938959624d247a3a42989c7cd1b55e5_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/bf2e893481774eb08ff0affbb9e7fa53_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/de4bf9b1d80d4cd481d8e6a2022e0834_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/ae4d14ec4cc74f8d92a4e25b0e917db3_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/744735a71e584194bc40e829c98ba8fb_16000_mono_16bit.wav 16000
openwakeword/notebooks/training_tutorial_data/positive/194362977cf44e99926950435

In [24]:
!pip uninstall -y piper-tts piper-phonemize piper-phonemize-fix
!pip install -q piper-phonemize-fix
!pip install -q --no-deps piper-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 49.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 8.6 MB/s eta 0:00:00


In [26]:
!pip install -q "setuptools<82" jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 35.6 MB/s eta 0:00:00


In [27]:
import piper
from piper import PiperVoice, SynthesisConfig

print("piper import OK")
print(PiperVoice)
print(SynthesisConfig)

piper import OK
<class 'piper.voice.PiperVoice'>
<class 'piper.config.SynthesisConfig'>


In [30]:
import pathlib
import sys

# 1. 准备 Piper ONNX 语音模型
!mkdir -p /content/piper-sample-generator/models

!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.onnx \
  https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx

!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.onnx.json \
  https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json

# 2. 重写 generate_samples.py，自动补 model 参数
shim = pathlib.Path("/content/piper-sample-generator/generate_samples.py")

shim.write_text(
    '''
from piper_sample_generator.__main__ import generate_samples_onnx as _generate_samples_onnx

DEFAULT_MODEL = "/content/piper-sample-generator/models/en_US-libritts_r-medium.onnx"

def generate_samples(*args, **kwargs):
    kwargs.setdefault("model", DEFAULT_MODEL)
    return _generate_samples_onnx(*args, **kwargs)
''',
    encoding="utf-8",
)

# 3. 加入 Python 搜索路径，并清掉旧 import 缓存
if "/content/piper-sample-generator" not in sys.path:
    sys.path.insert(0, "/content/piper-sample-generator")

sys.modules.pop("generate_samples", None)

# 4. 验证
import generate_samples
print("generate_samples wrapper OK")
print(generate_samples.generate_samples)

generate_samples wrapper OK
<function generate_samples at 0x7bb0df80d1c0>


In [32]:
import pathlib

target = pathlib.Path("/usr/local/lib/python3.12/dist-packages/dp/model/model.py")
text = target.read_text()

old = "torch.load(checkpoint_path, map_location=device)"
new = "torch.load(checkpoint_path, map_location=device, weights_only=False)"

if old in text:
    target.write_text(text.replace(old, new))
    print("patched dp/model/model.py")
elif new in text:
    print("already patched")
else:
    print("target line not found, inspect manually")

patched dp/model/model.py


In [33]:
!grep -n "torch.load" /usr/local/lib/python3.12/dist-packages/dp/model/model.py | head

306:    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)


In [34]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!PYTHONPATH=/content/piper-sample-generator:$PYTHONPATH \
  python openwakeword/openwakeword/train.py \
  --training_config my_model.yaml \
  --generate_clips

/usr/local/lib/python3.12/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
/usr/local/lib/python3.12/dist-packages/dp/model/model.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not Tr

In [57]:
from pathlib import Path

p = Path("/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py")
s = p.read_text()

if "import soundfile as sf" not in s:
    s = s.replace("import torchaudio", "import torchaudio\nimport soundfile as sf", 1)

s = s.replace("torchaudio.info(file_path)", "sf.info(file_path)")
s = s.replace("info.num_frames", "info.frames")
s = s.replace("info.sample_rate", "info.samplerate")

p.write_text(s)

print("patched:", p)
print("torchaudio.info still exists?", "torchaudio.info(file_path)" in p.read_text())

patched: /usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py
torchaudio.info still exists? False


In [58]:
# Step 2: Augment the generated clips

!PYTHONPATH=/content/piper-sample-generator:$PYTHONPATH \
  python openwakeword/openwakeword/train.py \
  --training_config my_model.yaml \
  --augment_clips

/usr/local/lib/python3.12/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_audiomen

In [ ]:
# Step 3: Train model

!PYTHONPATH=/content/piper-sample-generator:$PYTHONPATH \
  python openwakeword/openwakeword/train.py \
  --training_config my_model.yaml \
  --train_model

/usr/local/lib/python3.12/dist-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream
INFO:root:##################################################
Starting training sequence 1...
##################################################
Training:   8% 769/10000 [00:15<03:44, 41.15it/s]

In [60]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


ModuleNotFoundError: No module named 'onnx'

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!